# Zipping text with an LLM
### The notebook 3Blue1Brown's *Compression is Intelligence* series didn't show you

Grant Sanderson's series ([Part 1: Reinventing Entropy](https://www.youtube.com/watch?v=l6DKRf-fAAM), [Part 2: But what is cross-entropy?](https://www.youtube.com/watch?v=GlYgs6v2YfU)) makes one claim: **a model that predicts well is a model that compresses well.** An LLM's training loss is literally the number of bits it would need to store the text.

The videos stop at the theory. This notebook builds the actual compressor and puts it to four tests:

1. **Make a real file.** Use an LLM plus arithmetic coding to shrink a paragraph, then decompress it back byte for byte.
2. **Scoreboard.** LLM vs gzip vs xz on famous text, new text, Spanish, shuffled words and random noise.
3. **Bigger model, smaller file.** Does compression improve as models get bigger?
4. **Counting the model's size.** What happens when the decompressor's size is included?

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU*. Everything runs in about 5 minutes. It also works on CPU, just slower.

In [ ]:
# @title Setup
import math, time, json, zlib, lzma, random, gc, html, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAIN_MODEL = "Qwen/Qwen2.5-0.5B"          # small, multilingual base model (good at Spanish too)
SCALING_MODELS = ["gpt2", "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B"]   # add "Qwen/Qwen2.5-3B" if you have GPU memory to spare
os.makedirs("figures", exist_ok=True)
RESULTS = {"main_model": MAIN_MODEL, "device": DEVICE}
print("Device:", DEVICE, "| torch", torch.__version__)

# Chart style
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 11, "axes.titlesize": 14,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "legend.frameon": False,
})

## The idea in one paragraph

At every step an LLM outputs a probability for every possible next token. **Arithmetic coding** turns those probabilities into bits: a token the model gave probability *p* costs about **log₂(1/p) bits**. A token it was 99% sure about costs 0.01 bits. A token it gave 1 in 1,000 costs about 10 bits. The decoder runs *the same model*, gets *the same probabilities*, and reads the bits back into tokens. So the file size is the sum of the model's surprise, which is exactly the cross-entropy loss from Part 2.

The coder below is ~80 lines of plain Python using integer math, so encoder and decoder agree exactly.

In [ ]:
# @title Arithmetic coder (integer, exact)
import numpy as np

STATE_BITS = 64
FULL = 1 << STATE_BITS
HALF = FULL >> 1
QUARTER = HALF >> 1
MASK = FULL - 1
FREQ_TOTAL = 1 << 32  # probability resolution: every token gets at least 1 / 2^32


def probs_to_cumfreq(probs):
    """Turn a float probability vector into integer cumulative frequencies.
    Every token gets frequency >= 1, so nothing is ever 'impossible' to encode."""
    p = np.asarray(probs, dtype=np.float64)
    p = p / p.sum()
    freqs = np.floor(p * (FREQ_TOTAL - len(p))).astype(np.int64) + 1
    return np.concatenate(([0], np.cumsum(freqs)))


class ArithmeticEncoder:
    def __init__(self):
        self.low, self.high = 0, MASK
        self.pending = 0
        self.bits = []

    def _emit(self, bit):
        self.bits.append(bit)
        self.bits.extend([bit ^ 1] * self.pending)
        self.pending = 0

    def encode(self, cum, symbol):
        total = int(cum[-1])
        rng = self.high - self.low + 1
        c_lo, c_hi = int(cum[symbol]), int(cum[symbol + 1])
        self.high = self.low + c_hi * rng // total - 1
        self.low = self.low + c_lo * rng // total
        while ((self.low ^ self.high) & HALF) == 0:
            self._emit(self.low >> (STATE_BITS - 1))
            self.low = (self.low << 1) & MASK
            self.high = ((self.high << 1) & MASK) | 1
        while (self.low & ~self.high & QUARTER) != 0:
            self.pending += 1
            self.low = (self.low << 1) ^ HALF
            self.high = ((self.high ^ HALF) << 1) | HALF | 1

    def finish(self):
        self._emit(1)
        n = len(self.bits)
        return np.packbits(np.array(self.bits, dtype=np.uint8)).tobytes(), n


class ArithmeticDecoder:
    def __init__(self, data):
        self.bits = np.unpackbits(np.frombuffer(data, dtype=np.uint8)).tolist()
        self.pos = 0
        self.low, self.high = 0, MASK
        self.code = 0
        for _ in range(STATE_BITS):
            self.code = (self.code << 1) | self._read()

    def _read(self):
        b = self.bits[self.pos] if self.pos < len(self.bits) else 0
        self.pos += 1
        return b

    def decode(self, cum):
        total = int(cum[-1])
        rng = self.high - self.low + 1
        value = ((self.code - self.low + 1) * total - 1) // rng
        symbol = int(np.searchsorted(cum, value, side="right") - 1)
        c_lo, c_hi = int(cum[symbol]), int(cum[symbol + 1])
        self.high = self.low + c_hi * rng // total - 1
        self.low = self.low + c_lo * rng // total
        while ((self.low ^ self.high) & HALF) == 0:
            self.code = ((self.code << 1) & MASK) | self._read()
            self.low = (self.low << 1) & MASK
            self.high = ((self.high << 1) & MASK) | 1
        while (self.low & ~self.high & QUARTER) != 0:
            self.code = (self.code & HALF) | ((self.code << 1) & (MASK >> 1)) | self._read()
            self.low = (self.low << 1) ^ HALF
            self.high = ((self.high ^ HALF) << 1) | HALF | 1
        return symbol

In [ ]:
# @title Wrap a Hugging Face model as a "next-token probability machine"
class LM:
    def __init__(self, name, dtype=torch.float32):
        self.name = name
        self.tok = AutoTokenizer.from_pretrained(name)
        try:
            self.model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype)
        except TypeError:
            self.model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
        self.model = self.model.to(DEVICE).eval()
        self.eos = self.tok.eos_token_id          # also used as the "start of text" context
        self.n_params = sum(p.numel() for p in self.model.parameters())

    def encode_text(self, text):
        ids = self.tok.encode(text, add_special_tokens=False)
        assert self.decode_ids(ids) == text, "tokenizer does not round-trip this text"
        return ids

    def decode_ids(self, ids):
        return self.tok.decode(ids, clean_up_tokenization_spaces=False)

    def token_str(self, i):
        return self.decode_ids([i])

    # Step-by-step interface used by the compressor. Encoder and decoder call exactly
    # the same sequence of operations, so they see bit-identical probabilities.
    def start(self):
        self.past = None
        return self._step(self.eos)

    def next(self, token_id):
        return self._step(token_id)

    @torch.inference_mode()
    def _step(self, token_id):
        out = self.model(input_ids=torch.tensor([[token_id]], device=DEVICE),
                         past_key_values=self.past, use_cache=True)
        self.past = out.past_key_values
        return torch.softmax(out.logits[0, -1].double(), -1).cpu().numpy()

    # Fast path: one forward pass gives the bits for every token (= cross-entropy).
    @torch.inference_mode()
    def analyze(self, text, k=3):
        ids = self.encode_text(text)
        x = torch.tensor([[self.eos] + ids], device=DEVICE)
        logp = torch.log_softmax(self.model(x).logits[0, :-1].float(), -1)
        bits = (-logp[torch.arange(len(ids)), torch.tensor(ids)] / math.log(2)).cpu().numpy()
        topk = logp.topk(k, -1).indices.cpu().tolist()
        return ids, bits, topk

    def free(self):
        del self.model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

lm = LM(MAIN_MODEL)
print(f"Loaded {MAIN_MODEL}: {lm.n_params/1e6:.0f}M parameters")

## The texts

- **Famous (EN):** the opening of *Pride and Prejudice*. The model has almost certainly read it many times.
- **Famous (ES):** the opening of *Don Quijote*.
- **New (EN / ES):** a paragraph written for this notebook, so no model has seen it. The Spanish one is a translation of the English one, so we can compare languages fairly.
- **Shuffled:** the new English paragraph with its words in random order. Same words, no grammar.
- **Random:** random letters and spaces. Nothing to predict.
- **Yours:** paste anything you wrote recently (optional).

In [ ]:
# @title Texts
TEXTS = {}
TEXTS["Famous EN (Pride and Prejudice)"] = (
    "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife. "
    "However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so "
    "well fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of "
    "their daughters. \"My dear Mr. Bennet,\" said his lady to him one day, \"have you heard that Netherfield Park is let at last?\" "
    "Mr. Bennet replied that he had not. \"But it is,\" returned she; \"for Mrs. Long has just been here, and she told me all about it.\" "
    "Mr. Bennet made no answer. \"Do you not want to know who has taken it?\" cried his wife impatiently. "
    "\"You want to tell me, and I have no objection to hearing it.\" This was invitation enough.")
TEXTS["Famous ES (Don Quijote)"] = (
    "En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo de los de lanza en "
    "astillero, adarga antigua, rocín flaco y galgo corredor. Una olla de algo más vaca que carnero, salpicón las más noches, "
    "duelos y quebrantos los sábados, lantejas los viernes, algún palomino de añadidura los domingos, consumían las tres partes "
    "de su hacienda. El resto della concluían sayo de velarte, calzas de velludo para las fiestas, con sus pantuflos de lo mesmo, "
    "y los días de entresemana se honraba con su vellorí de lo más fino.")
TEXTS["New EN"] = (
    "The finance team at a mid-sized logistics company spent most of the first week of every month reconciling spreadsheets. "
    "Each regional office exported its own report, with its own column names and its own way of writing dates, and an analyst "
    "in the central office merged them by hand. Last year the company moved those reports into a shared data warehouse. The first "
    "version simply copied the old process: the same files, loaded every night, with the same inconsistencies. The second version "
    "introduced a single schema for shipments, invoices and returns, and a set of checks that rejects a file when its totals do not "
    "match the source system. The monthly close went from six days to two. The more important change was that the regional managers "
    "started trusting the numbers enough to use them in their weekly meetings, instead of keeping their own copies on the side.")
TEXTS["New ES"] = (
    "El equipo de finanzas de una empresa mediana de logística pasaba casi toda la primera semana de cada mes conciliando hojas de "
    "cálculo. Cada oficina regional exportaba su propio reporte, con sus propios nombres de columnas y su propia forma de escribir "
    "las fechas, y un analista en la oficina central los unía a mano. El año pasado la empresa llevó esos reportes a un repositorio "
    "de datos compartido. La primera versión simplemente copió el proceso anterior: los mismos archivos, cargados cada noche, con las "
    "mismas inconsistencias. La segunda versión introdujo un esquema único para envíos, facturas y devoluciones, y un conjunto de "
    "validaciones que rechaza un archivo cuando sus totales no coinciden con el sistema de origen. El cierre mensual pasó de seis días "
    "a dos. El cambio más importante fue que los gerentes regionales empezaron a confiar lo suficiente en las cifras como para usarlas "
    "en sus reuniones semanales, en lugar de mantener sus propias copias por aparte.")
_rng = random.Random(42)
_words = TEXTS["New EN"].split(" "); _rng.shuffle(_words)
TEXTS["Shuffled EN"] = " ".join(_words)
TEXTS["Random letters"] = "".join(_rng.choice("abcdefghijklmnopqrstuvwxyz     ") for _ in range(len(TEXTS["New EN"])))

YOUR_TEXT = ""   # @param {type:"string"}  paste any text here (optional). For line breaks, set TEXTS[YOUR_LABEL] = """...""" in a cell instead
YOUR_LABEL = "Yours"   # @param {type:"string"}  how this text is labeled in the charts, e.g. "Song lyrics (Geese, 2025)"
if YOUR_TEXT.strip():
    TEXTS[YOUR_LABEL] = YOUR_TEXT.strip()

for k, v in TEXTS.items():
    print(f"{k:34s} {len(v):5d} chars")

## Act 1: an actual compressed file made of predictions

We compress the new English paragraph with the LLM, decompress it, and check that we get back every byte. gzip and xz go on the same text for reference.

In [ ]:
# @title Compress → decompress round trip
def llm_compress(text, lm):
    ids = lm.encode_text(text) + [lm.eos]          # EOS marks the end of the message
    enc = ArithmeticEncoder()
    p = lm.start()
    for i, t in enumerate(ids):
        enc.encode(probs_to_cumfreq(p), t)
        if i < len(ids) - 1:
            p = lm.next(t)
    data, _ = enc.finish()
    return data

def llm_decompress(data, lm, max_tokens=100_000):
    dec = ArithmeticDecoder(data)
    p = lm.start(); ids = []
    for _ in range(max_tokens):
        t = dec.decode(probs_to_cumfreq(p))
        if t == lm.eos:
            break
        ids.append(t); p = lm.next(t)
    return lm.decode_ids(ids)

# Raw streams (no file headers/checksums), so short texts are compared fairly.
def gz(text):
    c = zlib.compressobj(9, zlib.DEFLATED, -15); return len(c.compress(text.encode()) + c.flush())
def xz(text):
    return len(lzma.compress(text.encode(), format=lzma.FORMAT_RAW,
                             filters=[{"id": lzma.FILTER_LZMA2, "preset": 9 | lzma.PRESET_EXTREME}]))

demo = TEXTS["New EN"]
t0 = time.time(); blob = llm_compress(demo, lm); t_enc = time.time() - t0
t0 = time.time(); back = llm_decompress(blob, lm); t_dec = time.time() - t0
t0 = time.time(); [zlib.compress(demo.encode(), 9) for _ in range(100)]; t_gz = (time.time() - t0) / 100

raw = len(demo.encode())
print(f"Original:        {raw:5d} bytes")
print(f"gzip:            {gz(demo):5d} bytes  ({raw/gz(demo):.1f}x)")
print(f"xz:              {xz(demo):5d} bytes  ({raw/xz(demo):.1f}x)")
print(f"LLM ({MAIN_MODEL.split('/')[-1]}): {len(blob):5d} bytes  ({raw/len(blob):.1f}x)")
print(f"\nDecompressed text identical to original: {back == demo}")
print(f"Speed: LLM {t_enc:.1f}s to compress, {t_dec:.1f}s to decompress | gzip {t_gz*1e6:.0f} µs  (~{t_enc/t_gz:,.0f}x slower)")
print("\nThe whole paragraph as LLM-compressed bytes:\n", blob.hex(" "))

RESULTS["act1"] = dict(raw_bytes=raw, gzip_bytes=gz(demo), xz_bytes=xz(demo), llm_bytes=len(blob),
                       roundtrip_ok=back == demo, llm_compress_s=round(t_enc, 2), llm_decompress_s=round(t_dec, 2),
                       gzip_s=t_gz, slowdown_x=round(t_enc / t_gz))

### Where do the bits go?

Each token is shaded by how many bits it cost, meaning how surprised the model was. Light means predictable (almost free), dark means surprising. This is Shannon's guessing game from Part 1, played by a machine.

📸 **Good screenshot for a post.**

In [ ]:
# @title Surprise heatmap
def surprise_html(text, lm, title):
    ids, bits, topk = lm.analyze(text)
    cap = 16
    spans = []
    for i, b in zip(ids, bits):
        s = min(b, cap) / cap
        # single-hue sequential ramp: near-white -> deep blue
        r, g, bl = [int(round(a + (c - a) * s)) for a, c in zip((244, 248, 253), (21, 64, 132))]
        fg = "#ffffff" if s > 0.45 else "#0b0b0b"
        tok = html.escape(lm.token_str(i))
        spans.append(f'<span title="{b:.1f} bits" style="background:rgb({r},{g},{bl});color:{fg};'
                     f'padding:2px 0;border-radius:3px;line-height:2.1">{tok}</span>')
    legend = "".join(f'<span style="display:inline-block;width:34px;height:12px;background:rgb('
                     + ",".join(str(int(round(a + (c - a) * s))) for a, c in zip((244, 248, 253), (21, 64, 132)))
                     + ')"></span>' for s in np.linspace(0, 1, 8))
    total = bits.sum()
    return f'''<div style="font-family:Inter,Helvetica,Arial,sans-serif;max-width:760px;background:#fcfcfb;padding:22px 26px;border-radius:10px;border:1px solid #e4e3df">
<div style="font-size:18px;font-weight:700;color:#0b0b0b">{title}</div>
<div style="font-size:13px;color:#52514e;margin:4px 0 14px">{MAIN_MODEL} · {len(ids)} tokens · {total:,.0f} bits total ({total/len(text):.2f} bits per character, gzip needs {gz(text)*8/len(text):.2f})</div>
<div style="font-size:15px;font-family:ui-monospace,Menlo,monospace;white-space:pre-wrap;overflow-wrap:anywhere;word-break:break-word">{"".join(spans)}</div>
<div style="font-size:12px;color:#52514e;margin-top:14px;display:flex;align-items:center;gap:8px">0 bits {legend} {cap}+ bits (hover a token for its cost)</div></div>'''

display(HTML(surprise_html(TEXTS["New EN"], lm, "Bits per token: new paragraph")))
display(HTML(surprise_html(TEXTS["Famous EN (Pride and Prejudice)"], lm, "Bits per token: Pride and Prejudice (opening)")))

In [ ]:
# @title The most and least surprising tokens (with the model's top guesses)
ids, bits, topk = lm.analyze(TEXTS["New EN"])
order = np.argsort(bits)
def row(j):
    guesses = ", ".join(repr(lm.token_str(g)) for g in topk[j])
    ctx = lm.decode_ids(ids[max(0, j - 6):j])
    return f"{bits[j]:6.2f} bits  …{ctx!r:>40} → {lm.token_str(ids[j])!r:14}  model guessed: {guesses}"
print("MOST SURPRISING")
for j in order[::-1][:8]: print(row(j))
print("\nALMOST FREE")
for j in order[:8]: print(row(j))
RESULTS["surprises"] = [dict(token=lm.token_str(ids[j]), bits=round(float(bits[j]), 2),
                             guesses=[lm.token_str(g) for g in topk[j]]) for j in order[::-1][:8]]
RESULTS["share_tokens_under_1bit"] = round(float((bits < 1).mean()), 3)
RESULTS["share_bits_from_top10pct_tokens"] = round(float(np.sort(bits)[::-1][: max(1, len(bits)//10)].sum() / bits.sum()), 3)
print(f"\n{RESULTS['share_tokens_under_1bit']:.0%} of tokens cost less than 1 bit.")
print(f"The 10% most surprising tokens account for {RESULTS['share_bits_from_top10pct_tokens']:.0%} of the file.")

## Act 2: the scoreboard

Bits per character (lower is better). Plain text uses 8. We use the fast path (one forward pass) because Act 1 showed the real coder lands within a couple of bytes of it.

Note: gzip and xz are measured as raw streams (no file headers) to be fair on short texts. They still do better on big files because they learn repeats as they go, which is why Act 4 uses their big-data numbers.

In [ ]:
# @title Scoreboard
def llm_bits(text, lm):
    ids, bits, _ = lm.analyze(text)
    return float(bits.sum()), len(ids)

rows = []
for name, text in TEXTS.items():
    b, ntok = llm_bits(text, lm)
    rows.append(dict(text=name, chars=len(text), tokens=ntok, gzip_bpc=gz(text) * 8 / len(text),
                     xz_bpc=xz(text) * 8 / len(text), llm_bpc=b / len(text)))
    rows[-1]["llm_vs_gzip_x"] = rows[-1]["gzip_bpc"] / rows[-1]["llm_bpc"]

print(f"{'text':34s} {'chars':>5} {'tokens':>6} {'gzip':>6} {'xz':>6} {'LLM':>6} {'LLM beats gzip by':>18}")
for r in rows:
    print(f"{r['text']:34s} {r['chars']:5d} {r['tokens']:6d} {r['gzip_bpc']:6.2f} {r['xz_bpc']:6.2f} {r['llm_bpc']:6.2f} {r['llm_vs_gzip_x']:17.1f}x")
RESULTS["scoreboard"] = [{k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()} for r in rows]

# Chart 1: grouped bars, bits per character
fig, ax = plt.subplots(figsize=(10, 5.6))
names = [r["text"] for r in rows][::-1]
y = np.arange(len(names)); h = 0.26
series = [("gzip", "gzip_bpc", ORANGE), ("xz", "xz_bpc", AQUA), (f"LLM ({MAIN_MODEL.split('/')[-1]})", "llm_bpc", BLUE)]
for k, (label, key, col) in enumerate(series):
    vals = [r[key] for r in rows][::-1]
    ax.barh(y + (1 - k) * h, vals, height=h - 0.03, color=col, label=label)
    if key == "llm_bpc":
        for yy, v in zip(y + (1 - k) * h, vals):
            ax.text(v + 0.08, yy, f"{v:.2f}", va="center", fontsize=9, color=INK)
ax.set_yticks(y); ax.set_yticklabels(names); ax.grid(axis="y", visible=False)
ax.axvline(8, color=INK2, lw=1, ls=(0, (3, 3))); ax.text(8.05, len(names) - 0.55, "plain text\n(8 bits)", fontsize=9, color=INK2, va="top")
ax.set_xlabel("bits per character (lower = better compression)")
ax.set_title("Bits per character by text and compressor")
ax.legend(loc="lower right", ncol=3, bbox_to_anchor=(1, -0.2))
plt.tight_layout(); plt.savefig("figures/1_scoreboard.png", dpi=200, bbox_inches="tight"); plt.show()

# Chart 2: the memorization gap, as one number per text
fig, ax = plt.subplots(figsize=(10, 4.6))
vals = [r["llm_vs_gzip_x"] for r in rows][::-1]
ax.barh(y, vals, height=0.6, color=BLUE)
for yy, v in zip(y, vals):
    ax.text(v + 0.05, yy, f"{v:.1f}x", va="center", fontsize=10, color=INK)
ax.axvline(1, color=INK2, lw=1); ax.text(1.03, -0.75, "no better than gzip", fontsize=9, color=INK2)
ax.set_yticks(y); ax.set_yticklabels(names); ax.grid(axis="y", visible=False)
ax.set_xlabel("how many times smaller the LLM's file is vs gzip")
ax.set_title("Compression vs gzip, by text")
plt.tight_layout(); plt.savefig("figures/2_memorization_gap.png", dpi=200, bbox_inches="tight"); plt.show()

## Act 3: bigger model, smaller file

If compression tracks intelligence, bigger and better models should compress new text better. We score the same two new paragraphs (English and Spanish) with models of different sizes. GPT-2 (2019, English-only training data) is included as the old baseline.

In [ ]:
# @title Model size vs compression
lm.free(); del lm
scaling = []
for name in SCALING_MODELS:
    m = LM(name)
    for lang in ["New EN", "New ES"]:
        b, ntok = llm_bits(TEXTS[lang], m)
        scaling.append(dict(model=name, params_m=round(m.n_params / 1e6), text=lang,
                            bpc=b / len(TEXTS[lang]), tokens=ntok))
    print(f"{name:22s} {m.n_params/1e6:6.0f}M  EN {scaling[-2]['bpc']:.2f} bpc ({scaling[-2]['tokens']} tokens)"
          f"  ES {scaling[-1]['bpc']:.2f} bpc ({scaling[-1]['tokens']} tokens)")
    m.free(); del m
RESULTS["scaling"] = [{k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()} for r in scaling]

fig, ax = plt.subplots(figsize=(9, 4.8))
labels = [f"{n.split('/')[-1]}\n{next(r['params_m'] for r in scaling if r['model']==n)}M params" for n in SCALING_MODELS]
x = np.arange(len(SCALING_MODELS)); w = 0.36
for k, (lang, col, lab) in enumerate([("New EN", BLUE, "English"), ("New ES", ORANGE, "Spanish")]):
    vals = [next(r["bpc"] for r in scaling if r["model"] == n and r["text"] == lang) for n in SCALING_MODELS]
    ax.bar(x + (k - 0.5) * w, vals, width=w - 0.03, color=col, label=lab)
    for xx, v in zip(x + (k - 0.5) * w, vals):
        ax.text(xx, v + 0.04, f"{v:.2f}", ha="center", fontsize=9, color=INK)
g = gz(TEXTS["New EN"]) * 8 / len(TEXTS["New EN"])
ax.axhline(g, color=INK2, lw=1, ls=(0, (3, 3))); ax.text(-0.45, g + 0.06, f"gzip on the same text ({g:.1f})", ha="left", fontsize=9, color=INK2)
ymax = max([g] + [r["bpc"] for r in scaling]) * 1.18
ax.set_ylim(0, ymax)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.grid(axis="x", visible=False)
ax.set_ylabel("bits per character (lower = better)")
ax.set_title("Bits per character by model size (new paragraph)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.tight_layout(); plt.savefig("figures/3_model_size.png", dpi=200, bbox_inches="tight"); plt.show()

## Act 4: counting the model's size

To decompress, the other person needs the model too. A fair comparison counts the decompressor's size, which is how the [Hutter Prize](http://prize.hutter1.net/) scores entries. gzip's program is ~100 KB. A 0.5B-parameter model is ~1 GB in 16-bit weights.

For the classic compressors we use their well-known rates on large English text (enwik8, 100 MB of Wikipedia: gzip ≈ 2.9, xz ≈ 2.0 bits per character) instead of our short-text numbers, which are penalized by headers. For the LLM we use its rate on the **new** English paragraph, so memorization doesn't flatter it.

In [ ]:
# @title When does the LLM pay for itself?
GZIP_BPC_LARGE, XZ_BPC_LARGE = 2.92, 1.99       # enwik8 benchmark rates (approx.)
PROGRAM_BYTES = 100e3                            # rough size of the gzip / xz programs
main_row = next(r for r in scaling if r["model"] == MAIN_MODEL and r["text"] == "New EN")
LLM_BPC = main_row["bpc"]; MODEL_BYTES = main_row["params_m"] * 1e6 * 2   # 16-bit weights

N = np.logspace(3, 12, 400)                      # characters of text: 1 KB ... 1 TB
total = lambda bpc, overhead: overhead + N * bpc / 8
curves = [("gzip", GZIP_BPC_LARGE, PROGRAM_BYTES, ORANGE), ("xz", XZ_BPC_LARGE, PROGRAM_BYTES, AQUA),
          (f"LLM + its weights ({MAIN_MODEL.split('/')[-1]})", LLM_BPC, MODEL_BYTES, BLUE)]
breakeven = {}
for label, bpc, ov, _ in curves[:2]:
    breakeven[label] = (MODEL_BYTES - ov) * 8 / (bpc - LLM_BPC) if bpc > LLM_BPC else float("inf")

fig, ax = plt.subplots(figsize=(9, 5))
for label, bpc, ov, col in curves:
    ax.loglog(N, total(bpc, ov), color=col, lw=2, label=label)
be = breakeven["xz"]
if np.isfinite(be):
  ax.plot([be], [MODEL_BYTES + be * LLM_BPC / 8], "o", ms=9, color=BLUE, mec="#fcfcfb", mew=2)
  ax.annotate(f"breaks even with xz at\n~{be/1e9:.1f} GB of text", (be, MODEL_BYTES + be * LLM_BPC / 8),
              xytext=(-150, 25), textcoords="offset points", fontsize=10, color=INK,
              arrowprops=dict(arrowstyle="-", color=INK2, lw=1))
def _fmt_bytes(v, _):
    for unit, f in [("TB", 1e12), ("GB", 1e9), ("MB", 1e6), ("KB", 1e3)]:
        if v >= f: return f"{v/f:g} {unit}"
    return f"{v:g} B"
from matplotlib.ticker import FuncFormatter, LogLocator
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_locator(LogLocator(base=1000)); axis.set_major_formatter(FuncFormatter(_fmt_bytes))
    axis.set_minor_locator(LogLocator(base=10, subs=(1.0,))); axis.set_minor_formatter(FuncFormatter(lambda v, _: ""))
ax.set_xlabel("amount of text to store (characters ≈ bytes)")
ax.set_ylabel("total bytes you have to ship")
ax.set_title("Total size including the decompressor")
ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig("figures/4_honest_bill.png", dpi=200, bbox_inches="tight"); plt.show()

RESULTS["honest_bill"] = dict(llm_bpc_new_en=round(LLM_BPC, 3), model_gb_fp16=round(MODEL_BYTES / 1e9, 2),
                              breakeven_vs_gzip_gb=round(breakeven["gzip"] / 1e9, 2),
                              breakeven_vs_xz_gb=round(breakeven["xz"] / 1e9, 2))
print(RESULTS["honest_bill"])

## Bonus: cover images for your own text

Two 1920x1080 images built from the text in `TEXTS[YOUR_LABEL]`, with the words hidden (useful for song lyrics or anything you should not republish). Paste the text with its real line breaks so repeated lines can be detected.

In [ ]:
# @title Cover image: the song as a "surprise waveform" (words hidden, 1920x1080)
from matplotlib.colors import LinearSegmentedColormap
if "lm" not in globals():
    lm = LM(MAIN_MODEL)
SONG_TITLE = f'{YOUR_LABEL}, as an LLM hears it'   # edit freely
song = TEXTS[YOUR_LABEL]
ids, bits, _ = lm.analyze(song)
CAP = 16
b = np.minimum(bits, CAP)
cmap = LinearSegmentedColormap.from_list("bits", ["#c9d9f0", "#154084"])

# Which tokens belong to a line the song already sang before? (needs the lyrics pasted with line breaks)
line_of, line, seen, repeat = [], 0, set(), []
for i in ids:
    line_of.append(line); line += lm.token_str(i).count("\n")
lines = song.split("\n")
first_time = {}
for n, l in enumerate(lines):
    key = l.strip().lower()
    first_time[n] = bool(key) and key not in seen
    if key: seen.add(key)
repeat = np.array([not first_time.get(n, True) for n in line_of])

fig = plt.figure(figsize=(16, 9))
ax = fig.add_axes([0.06, 0.22, 0.88, 0.52])
x = np.arange(len(b))
ax.bar(x, b, width=0.8, color=cmap(b / CAP), linewidth=0)
ax.bar(x, -0.5 * b, width=0.8, color=cmap(b / CAP), linewidth=0, alpha=0.35)   # soft reflection
if repeat.any():
    # shade the stretches where the song repeats an earlier line
    start = None
    for k in range(len(repeat) + 1):
        on = k < len(repeat) and repeat[k]
        if on and start is None: start = k
        if not on and start is not None:
            ax.axvspan(start - 0.5, k - 0.5, color="#eda100", alpha=0.10, lw=0); start = None
ax.axhline(0, color="#fcfcfb", lw=1.2)
ax.set_xlim(-1, len(b)); ax.set_ylim(-CAP * 0.55, CAP * 1.05)
ax.axis("off")

total = bits.sum()
fig.text(0.06, 0.88, SONG_TITLE, fontsize=28, fontweight="bold", color=INK)
sub = (f"Every bar is one token of the lyrics; its height is how surprised {MAIN_MODEL} was (bits). "
       f"{len(ids)} tokens, {total/len(song):.2f} bits per character (gzip needs {gz(song)*8/len(song):.2f}).")
fig.text(0.06, 0.82, sub, fontsize=13.5, color=INK2)
if repeat.any():
    new_avg, rep_avg = bits[~repeat].mean(), bits[repeat].mean()
    fig.text(0.06, 0.12, f"Shaded: lines the song already sang. Average cost {rep_avg:.1f} bits per token, "
             f"vs {new_avg:.1f} the first time.", fontsize=14, color=INK)
    RESULTS["song_repeat_vs_new_bits"] = [round(float(rep_avg), 2), round(float(new_avg), 2)]
fig.text(0.06, 0.07, "Words hidden. Darker and taller = harder to predict.", fontsize=12, color=INK2)
plt.savefig("figures/0_cover_waveform.png", dpi=120)    # 1920 x 1080 px
plt.show()

In [ ]:
# @title Cover image: song heatmap (words hidden, 16:9 for LinkedIn)
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
if "lm" not in globals():
    lm = LM(MAIN_MODEL)
SONG_TITLE = f'Bits per token: {YOUR_LABEL}'   # edit freely
WRAP = 48          # max characters per row when the text has no (or very long) lines
song = TEXTS[YOUR_LABEL]
ids, bits, _ = lm.analyze(song)
CAP = 16
cmap = LinearSegmentedColormap.from_list("bits", ["#f4f8fd", "#154084"])

# One block per token (width = number of characters). New row at each line break,
# and wrap long rows at WRAP characters so the picture stays compact.
rows, row, used = [], [], 0
for i, b in zip(ids, bits):
    for k, part in enumerate(lm.token_str(i).split("\n")):
        if k > 0:
            rows.append(row); row, used = [], 0
        if part.strip():
            w = len(part)
            if used + w > WRAP and row:
                rows.append(row); row, used = [], 0
            row.append((w, float(b))); used += w
rows.append(row)
while rows and not rows[-1]: rows.pop()

width = max(sum(w for w, _ in r) for r in rows if r)
fig = plt.figure(figsize=(16, 9))                       # 16:9, LinkedIn cover format
ax = fig.add_axes([0.06, 0.20, 0.88, 0.58])
for y, r in enumerate(rows):
    x = 0
    for w, b in r:
        ax.add_patch(plt.Rectangle((x + 0.15, -y), w - 0.3, 0.72, color=cmap(min(b, CAP) / CAP), lw=0))
        x += w
ax.set_xlim(0, width); ax.set_ylim(-len(rows) + 0.1, 0.9)
ax.axis("off")
total = bits.sum()
fig.text(0.06, 0.90, SONG_TITLE, fontsize=26, fontweight="bold", color=INK)
fig.text(0.06, 0.84, f"{MAIN_MODEL} · {len(ids)} tokens · {total/len(song):.2f} bits per character "
         f"(gzip needs {gz(song)*8/len(song):.2f}) · each block is one token, darker = more surprising, words hidden",
         fontsize=13, color=INK2)
cax = fig.add_axes([0.30, 0.09, 0.40, 0.025])
cb = fig.colorbar(ScalarMappable(cmap=cmap, norm=plt.Normalize(0, CAP)), cax=cax, orientation="horizontal")
cb.set_label("bits per token (16+ capped)", color=INK2, fontsize=12); cb.outline.set_visible(False)
plt.savefig("figures/0_cover_song.png", dpi=120)        # 1920 x 1080 px
plt.show()

## Export

Copy the JSON below (it has every number from this run), and download `figures.zip` for the charts.

In [ ]:
# @title Results JSON + figures
print(json.dumps(RESULTS, ensure_ascii=False, default=float))
import shutil; shutil.make_archive("figures", "zip", "figures")
try:
    from google.colab import files; files.download("figures.zip")
except Exception:
    print("figures.zip is in the working directory")

## Caveats worth knowing

- **Same machine for both sides.** Encoder and decoder must get bit-identical probabilities. Here both run on the same runtime, so it works. Across different GPUs or library versions, tiny float differences can break decoding. Real tools like [ts_zip](https://bellard.org/ts_zip/) and [llama-zip](https://github.com/AlexBuz/llama-zip) deal with this explicitly.
- **Short texts.** gzip and xz learn as they go, so a few hundred bytes is their worst case even without headers. On big files they improve a lot (see Act 4), and the gap to the LLM narrows.
- **Memorization is not understanding.** A famous text compresses well partly because the model has stored it. This is also why benchmarks that compress well-known corpora can overstate how good a model is.

**Further reading:** Delétang et al., [*Language Modeling Is Compression*](https://arxiv.org/abs/2309.10668) (DeepMind, 2023), Shannon, *Prediction and Entropy of Printed English* (1951).